In [3]:
from typing import Optional

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, LlamaModel, LlamaConfig, LlamaForCausalLM


MODEL = "meta-llama/Llama-3.2-3B"

In [31]:
import torch.nn as nn

from transformers.masking_utils import create_causal_mask
from transformers.modeling_outputs import BaseModelOutputWithPast
from transformers.models.llama.configuration_llama import LlamaConfig


class AugmentedLlamaModel(LlamaModel):

    def __init__(self, config, insert_layer=0, virtual_token_count=None):
        super().__init__(config)

        # if insert_layer < 0 or insert_layer >= len(self.layers):
        #     raise ValueError(f"insert_layer must be between 0 and {len(self.layers) - 1}.")

        self.insert_layer = insert_layer
        self.virtual_token_count = virtual_token_count
        self.virtual_active = False
        self.virtual_tokens = None

    def new_virtual_tokens(self, init="zero", batch_size=1):
        if init == "zero":
            return torch.zeros(batch_size, self.config.hidden_size)
        # elif init == "random":
        #     pass
        else:
            raise ValueError(f'Unrecognized init method "{init}".')

    def forward(
        self,
        input_ids       = None,
        attention_mask  = None,
        position_ids    = None,
        past_key_values = None,
        inputs_embeds   = None,
        cache_position  = None,
        use_cache       = None,
        ### don't actually want to pass this in because it will mess with model.generate
        virtual_tokens  = None,
        **kwargs,
    ):
        if (input_ids is None) ^ (inputs_embeds is not None):
            raise ValueError("You must specify exactly one of input_ids or inputs_embeds")

        if self.virtual_token_count is not None and virtual_tokens is None:
            raise ValueError("You must provide virtual_tokens when virtual_token_count is configured.")
        elif self.virtual_token_count is None and virtual_tokens is not None:
            raise ValueError("Unexpected virtual_tokens. Ensure virtual_token_count is configured.")

        if inputs_embeds is None:
            inputs_embeds: torch.Tensor = self.embed_tokens(input_ids)

        ### construct expanded sequence with virtual tokens
        if virtual_tokens is not None:
            inputs_embeds = torch.cat([ virtual_tokens, inputs_embeds ], dim=-1)
            attention_mask      = torch.cat([ torch.zeroslike(virtual_tokens), attention_mask ], dim=-1)
            attention_mask_von  = torch.cat([ torch.oneslike(virtual_tokens),  attention_mask ], dim=-1)
        ###

        if use_cache and past_key_values is None:
            past_key_values = DynamicCache(config=self.config)

        if cache_position is None:
            past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0
            cache_position: torch.Tensor = torch.arange(
                past_seen_tokens, past_seen_tokens + inputs_embeds.shape[1], device=inputs_embeds.device
            )

        if position_ids is None:
            position_ids = cache_position.unsqueeze(0)

        # virtual_attention_mask = self.create_virtual_attention_mask(attention_mask)

        causal_mask = create_causal_mask(
            config=self.config,
            input_embeds=inputs_embeds,
            attention_mask=attention_mask,
            cache_position=cache_position,
            past_key_values=past_key_values,
            position_ids=position_ids,
        )

        if virtual_tokens is not None:
            causal_mask_von = create_causal_mask(
                config=self.config,
                input_embeds=inputs_embeds,
                attention_mask=attention_mask_von,
                cache_position=cache_position,
                past_key_values=past_key_values,
                position_ids=position_ids,
            )

        hidden_states = inputs_embeds
        position_embeddings = self.rotary_emb(hidden_states, position_ids)

        for i, decoder_layer in enumerate(self.layers[: self.config.num_hidden_layers]):
            mask = causal_mask if virtual_tokens is None or i < self.start_layer else causal_mask_von
            hidden_states = decoder_layer(
                hidden_states,
                attention_mask=mask,
                position_ids=position_ids,
                past_key_values=past_key_values,
                cache_position=cache_position,
                position_embeddings=position_embeddings,
                **kwargs,
            )

        hidden_states = self.norm(hidden_states)
        return BaseModelOutputWithPast(
            last_hidden_state=hidden_states,
            past_key_values=past_key_values,
        )

class AugmentedLlamaForCausalLM(LlamaForCausalLM):
    def __init__(self, config, **kwargs):
        super().__init__(config)
        self.model = AugmentedLlamaModel(config, **kwargs)
        self.vocab_size = config.vocab_size
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        # Initialize weights and apply final processing
        self.post_init()


In [29]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config = LlamaConfig.from_pretrained(MODEL)
model = AugmentedLlamaForCausalLM.from_pretrained(MODEL)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [30]:
import torch.nn.functional as F


text = "The sky is "
tokenized = tokenizer(text, return_tensors="pt")

output = model.generate(
    **tokenized,
    max_new_tokens=10,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

tokenizer.decode(output[0])

'<|begin_of_text|>The sky is 70% cloudy and the temperature is 65 degrees'

### Scratch

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

if False:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
    model.model.resize_token_embeddings(len(tokenizer))
    
    new_token_ids = tokenizer.convert_tokens_to_ids(["[PAD]"])
    
    if False:
        with torch.no_grad():
            for token_id in new_token_ids:
                model.model.get_input_embeddings().weight[token_id] = torch.zeros_like(model.model.get_input_embeddings().weight[token_id])
        print("Embedding of new token set to 0.")
    
    model.model.embed_tokens(torch.tensor(new_token_ids))

In [12]:
import torch.nn.functional as F

padding = "<|begin_of_text|><|begin_of_text|><|begin_of_text|>"
text = "Today is "

tokenized = tokenizer(padding + text, return_tensors="pt")
output = model(input_ids=tokenized["input_ids"], attention_mask=tokenized["attention_mask"])
padded = F.softmax(output.logits[:, -1, :], dim=-1)

tokenized = tokenizer(text, return_tensors="pt")
output = model(input_ids=tokenized["input_ids"], attention_mask=tokenized["attention_mask"])
bare = F.softmax(output.logits[:, -1, :], dim=-1)

print(padded, bare)

tensor([[6.8169e-10, 1.7396e-10, 6.6214e-08,  ..., 6.3789e-11, 6.3816e-11,
         6.3830e-11]], grad_fn=<SoftmaxBackward0>) tensor([[9.6671e-10, 6.2861e-10, 9.2969e-08,  ..., 4.0685e-11, 4.0703e-11,
         4.0719e-11]], grad_fn=<SoftmaxBackward0>)
